In [1]:
import ccxt
import pandas as pd
import numpy as np

exchange = ccxt.binance({
    "enableRateLimit": True,
    "options": {
        "defaultType": "future"   # ← THIS switches to futures
    }
})


In [2]:
markets = exchange.load_markets()

usdt_pairs = [
    symbol for symbol, market in markets.items()
    if market['quote'] == 'USDT'     # Must be USDT pair
    and market['contract']           # Must be derivative
    and market['swap']               # Must be perpetual (not quarterly)
    and market['active']             # Must be active
]

filtered_pairs = []

for symbol in usdt_pairs:
    print(f"scan {symbol}")
    ticker = exchange.fetch_ticker(symbol)
    if ticker['quoteVolume'] and ticker['quoteVolume'] > 5_000_000:
        filtered_pairs.append(symbol)

print(f"High-liquidity USDT futures: {len(filtered_pairs)}")



print(f"Total USDT perpetual futures pairs: {len(usdt_pairs)}")



scan BTC/USDT:USDT
scan ETH/USDT:USDT
scan BCH/USDT:USDT
scan XRP/USDT:USDT
scan LTC/USDT:USDT
scan TRX/USDT:USDT
scan ETC/USDT:USDT
scan LINK/USDT:USDT
scan XLM/USDT:USDT
scan ADA/USDT:USDT
scan XMR/USDT:USDT
scan DASH/USDT:USDT
scan ZEC/USDT:USDT
scan XTZ/USDT:USDT
scan BNB/USDT:USDT
scan ATOM/USDT:USDT
scan ONT/USDT:USDT
scan IOTA/USDT:USDT
scan BAT/USDT:USDT
scan VET/USDT:USDT
scan NEO/USDT:USDT
scan QTUM/USDT:USDT
scan IOST/USDT:USDT
scan THETA/USDT:USDT
scan ALGO/USDT:USDT
scan ZIL/USDT:USDT
scan KNC/USDT:USDT
scan ZRX/USDT:USDT
scan COMP/USDT:USDT
scan DOGE/USDT:USDT
scan KAVA/USDT:USDT
scan BAND/USDT:USDT
scan RLC/USDT:USDT
scan SNX/USDT:USDT
scan DOT/USDT:USDT
scan YFI/USDT:USDT
scan CRV/USDT:USDT
scan TRB/USDT:USDT
scan RUNE/USDT:USDT
scan SUSHI/USDT:USDT
scan EGLD/USDT:USDT
scan SOL/USDT:USDT
scan ICX/USDT:USDT
scan STORJ/USDT:USDT
scan UNI/USDT:USDT
scan AVAX/USDT:USDT
scan ENJ/USDT:USDT
scan KSM/USDT:USDT
scan NEAR/USDT:USDT
scan AAVE/USDT:USDT
scan FIL/USDT:USDT
scan RSR/

In [3]:
def get_ohlcv(symbol, timeframe='15m', limit=500):
    ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)

    df = pd.DataFrame(ohlcv, columns=[
        "timestamp","open","high","low","close","volume"
    ])

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    
    return df


In [4]:
def rma(series, period):
    rma = series.copy()
    rma.iloc[:period] = series.iloc[:period].mean()

    alpha = 1 / period
    for i in range(period, len(series)):
        rma.iloc[i] = (series.iloc[i] * alpha) + (rma.iloc[i-1] * (1 - alpha))

    return rma


In [5]:
def add_supertrend(df, period=10, multiplier=3.0):

    df = df.copy()

    # ======================
    # TRUE RANGE
    # ======================
    df['prev_close'] = df['close'].shift(1)

    tr1 = df['high'] - df['low']
    tr2 = (df['high'] - df['prev_close']).abs()
    tr3 = (df['low'] - df['prev_close']).abs()

    df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    # ======================
    # WILDER ATR (MATCH PINE)
    # ======================
    df['atr'] = rma(df['tr'], period)

    # ======================
    # BASIC BANDS
    # ======================
    hl2 = (df['high'] + df['low']) / 2

    df['upperband'] = hl2 + multiplier * df['atr']
    df['lowerband'] = hl2 - multiplier * df['atr']

    # ======================
    # FINAL BANDS (Recursive)
    # ======================
    df['final_upperband'] = df['upperband']
    df['final_lowerband'] = df['lowerband']

    for i in range(1, len(df)):

        if df['close'].iloc[i-1] > df['final_upperband'].iloc[i-1]:
            df.at[df.index[i], 'final_upperband'] = df['upperband'].iloc[i]
        else:
            df.at[df.index[i], 'final_upperband'] = min(
                df['upperband'].iloc[i],
                df['final_upperband'].iloc[i-1]
            )

        if df['close'].iloc[i-1] < df['final_lowerband'].iloc[i-1]:
            df.at[df.index[i], 'final_lowerband'] = df['lowerband'].iloc[i]
        else:
            df.at[df.index[i], 'final_lowerband'] = max(
                df['lowerband'].iloc[i],
                df['final_lowerband'].iloc[i-1]
            )

    # ======================
    # DIRECTION (MATCH TRADINGVIEW)
    # IMPORTANT:
    # direction < 0  → UP trend
    # direction > 0  → DOWN trend
    # ======================
    df['direction'] = 1

    for i in range(1, len(df)):

        if df['close'].iloc[i] > df['final_upperband'].iloc[i-1]:
            df.at[df.index[i], 'direction'] = -1

        elif df['close'].iloc[i] < df['final_lowerband'].iloc[i-1]:
            df.at[df.index[i], 'direction'] = 1

        else:
            df.at[df.index[i], 'direction'] = df['direction'].iloc[i-1]

    # ======================
    # FLIP DETECTION (EXACTLY LIKE PINE)
    # ======================
    df['direction_change'] = df['direction'] - df['direction'].shift(1)

    df['flip_up'] = df['direction_change'] < 0     # Down → Up
    df['flip_down'] = df['direction_change'] > 0   # Up → Down

    return df


In [6]:
def check_supertrend_flip(df):

    # Use last CLOSED candle
    last = df.iloc[-2]

    if last['flip_up']:
        return {
            "direction": "UP",
            "price": last['low'],  # exactly like Pine
            "time": last['timestamp']
        }

    elif last['flip_down']:
        return {
            "direction": "DOWN",
            "price": last['high'],  # exactly like Pine
            "time": last['timestamp']
        }

    return None


In [7]:
timeframes = ['15m', '1h', '4h', '1d']

def analyze_symbol(symbol):
    symbol_signals = {}

    for tf in timeframes:
        try:
            df = get_ohlcv(symbol, tf, 500)
            df = add_supertrend(df)

            result = check_supertrend_flip(df)

            if result:
                result['timeframe'] = tf  # attach timeframe
                symbol_signals[tf] = result

        except Exception as e:
            print(f"Error {symbol} {tf}: {e}")

    return symbol_signals



In [8]:
def scan_market(pairs):

    print("\n=== LIVE SUPERTREND FLIP SCAN (MULTI-TF) ===\n")

    signals = {}

    for symbol in pairs:
        print(f"scanning : {symbol}")

        result = analyze_symbol(symbol)

        if result:   # if at least one timeframe triggered

            signals[symbol] = result

            print(f"\n🔥 {symbol}")

            for tf, info in result.items():
                print(
                    f"  TF: {tf} → {info['direction']} "
                    f"| Price: {info['price']} "
                    f"| Time: {info['time']}"
                )

            print("-" * 50)

    return signals


In [9]:
signals = scan_market(filtered_pairs)




=== LIVE SUPERTREND FLIP SCAN (MULTI-TF) ===

scanning : BTC/USDT:USDT
scanning : ETH/USDT:USDT
scanning : BCH/USDT:USDT
scanning : XRP/USDT:USDT
scanning : LTC/USDT:USDT
scanning : TRX/USDT:USDT
scanning : ETC/USDT:USDT
scanning : LINK/USDT:USDT
scanning : XLM/USDT:USDT
scanning : ADA/USDT:USDT

🔥 ADA/USDT:USDT
  TF: 15m → DOWN | Price: 0.2765 | Time: 2026-02-20 13:15:00
--------------------------------------------------
scanning : XMR/USDT:USDT
scanning : DASH/USDT:USDT
scanning : ZEC/USDT:USDT
scanning : XTZ/USDT:USDT
scanning : BNB/USDT:USDT
scanning : ATOM/USDT:USDT

🔥 ATOM/USDT:USDT
  TF: 1h → UP | Price: 2.296 | Time: 2026-02-20 12:00:00
--------------------------------------------------
scanning : VET/USDT:USDT
scanning : ALGO/USDT:USDT
scanning : ZIL/USDT:USDT
scanning : COMP/USDT:USDT
scanning : DOGE/USDT:USDT
scanning : SNX/USDT:USDT
scanning : DOT/USDT:USDT
scanning : CRV/USDT:USDT
scanning : TRB/USDT:USDT
scanning : SOL/USDT:USDT
scanning : UNI/USDT:USDT
scanning : AVAX/U